In [ ]:
"""
Sensitivity of the clustering to the autoencoder latent dimension.
No patient data are included; set the input paths to the local label files.
"""

import pandas as pd
import numpy as np
from sklearn.metrics import adjusted_rand_score
from itertools import combinations

files = {f"latent_{d}": f"cluster_labels_latent_{d}_kmeans.csv"
         for d in range(10, 101, 10)}

dfs = {}
for name, filepath in files.items():
    df = pd.read_csv(filepath).rename(columns={"Unnamed: 0": "patient_id",
                                               "Cluster": name})
    dfs[name] = df

merged = list(dfs.values())[0][["patient_id", list(files.keys())[0]]]
for name, df in list(dfs.items())[1:]:
    merged = merged.merge(df[["patient_id", name]], on="patient_id")

results = []
for dim1, dim2 in combinations(files.keys(), 2):
    ari = adjusted_rand_score(merged[dim1], merged[dim2])
    results.append({"Comparison": f"{dim1} vs {dim2}", "ARI": round(ari, 3)})

results_df = pd.DataFrame(results).sort_values("ARI", ascending=False)
print(results_df.to_string(index=False))